# Shoplifting Detection — Teacher → Student Knowledge Distillation

This Colab notebook trains a stronger VideoMAE-Small teacher on the binary normal/shoplifting task, then distills its logits into a lightweight MoViNet-A0 student. The final deployment artifact contains only the MoViNet student.

**Run mode:** GPU. Video decoding is CPU-side; model training/inference runs on CUDA. The notebook downloads only the selected UCF-Crime split videos rather than the full archive.


### Dataset source fix
The current `jinmang2/ucf_crime` Hugging Face repository no longer stores individual MP4 files at paths such as `Shoplifting/Shoplifting027_x264.mp4`; those direct entries were removed and the videos are now packaged in ZIP archives. The downloader below targets the current archive layout and extracts only selected clips. citeturn3search6turn1search9


In [ ]:
!nvidia-smi -L
!pip -q install -U "transformers>=4.45" "accelerate>=0.34" "huggingface_hub>=0.25" "decord>=0.6.0" "scikit-learn>=1.4" "pandas>=2.0" "matplotlib>=3.8" "seaborn>=0.13" "tqdm>=4.66"


In [ ]:
import os, json, random, math, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from huggingface_hub import hf_hub_download
from decord import VideoReader, cpu
from transformers import AutoImageProcessor, AutoModelForVideoClassification
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, classification_report, confusion_matrix,
                             roc_curve, precision_recall_curve)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

assert torch.cuda.is_available(), "Enable a Colab GPU: Runtime → Change runtime type → GPU"
DEVICE = torch.device("cuda")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
random.seed(42); np.random.seed(42); torch.manual_seed(42); torch.cuda.manual_seed_all(42)

ROOT = Path("/content/shoplifting_distillation")
VIDEO_DIR = ROOT / "videos"
FIG_DIR = ROOT / "figures"
CKPT_DIR = ROOT / "checkpoints"
for p in (VIDEO_DIR, FIG_DIR, CKPT_DIR): p.mkdir(parents=True, exist_ok=True)

UCF_REPO = "jinmang2/ucf_crime"
# IMPORTANT: the current HF repo stores UCF-Crime videos inside ZIP archives; the old
# direct paths such as Shoplifting/Shoplifting027_x264.mp4 were removed from the repo.
# See: https://huggingface.co/datasets/jinmang2/ucf_crime/tree/main
UCF_ARCHIVES = {
    "anomaly1": "Anomaly-Videos-Part-1.zip",
    "anomaly2": "Anomaly-Videos-Part-2.zip",
    "anomaly3": "Anomaly-Videos-Part-3.zip",
    "anomaly4": "Anomaly-Videos-Part-4.zip",
    "normal_test": "Testing_Normal_Videos.zip",
    "normal_train1": "Training-Normal-Videos-Part-1.zip",
    "normal_train2": "Training-Normal-Videos-Part-2.zip",
}
TRAIN_SPLIT = "UCF_Crimes-Train-Test-Split/Action_Recognition_splits/train_001.txt"
TEST_SPLIT = "UCF_Crimes-Train-Test-Split/Action_Recognition_splits/test_001.txt"
TEACHER_ID = "MCG-NJU/videomae-small-finetuned-kinetics"
STUDENT_ID = "kfkas/movinet-a0-stream-pytorch"
LABEL2ID = {"normal":0, "shoplifting":1}; ID2LABEL = {0:"normal",1:"shoplifting"}
NUM_FRAMES = 16
STUDENT_SIZE = 172
TEACHER_SIZE = 224
TEACHER_EPOCHS = 2
DISTILL_EPOCHS = 5
TEACHER_LR = 1e-5
STUDENT_LR = 2e-4
WEIGHT_DECAY = 1e-4
KD_TEMPERATURE = 3.0
KD_ALPHA = 0.5
KD_BETA = 0.5
VAL_CLIPS = 3
MAX_TRAIN_VIDEOS = 0  # set to e.g. 40 for a smoke test
MAX_VAL_VIDEOS = 0    # set to e.g. 20 for a smoke test
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
TRAIN_BATCH = 4 if GPU_GB >= 20 else (2 if GPU_GB >= 10 else 1)
TEACHER_BATCH = 2 if GPU_GB >= 20 else 1
NUM_WORKERS = 2
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {GPU_GB:.1f} GB")
print(f"student batch={TRAIN_BATCH}, teacher batch={TEACHER_BATCH}")


## 1. Build a balanced manifest and download videos

The UCF-Crime split contains explicit `Shoplifting/` examples and normal surveillance videos. We balance the two classes at the manifest level and validate every downloaded file before training.


In [ ]:
def read_split(path):
    local = hf_hub_download(repo_id=UCF_REPO, filename=path, repo_type="dataset")
    rows=[]
    for line in Path(local).read_text(errors="ignore").splitlines():
        line=line.strip()
        if not line: continue
        rel=line.split()[0].replace("\\", "/")
        if rel.startswith("Shoplifting/"): label=1
        elif "Normal_Videos" in rel: label=0
        else: continue
        rows.append((rel,label))
    return rows

train_rows = read_split(TRAIN_SPLIT)
val_rows = read_split(TEST_SPLIT)
def balance(rows, max_n=0):
    rng=random.Random(42)
    pos=[x for x in rows if x[1]==1]; neg=[x for x in rows if x[1]==0]
    n=min(len(pos),len(neg))
    if max_n: n=min(n,max_n)
    rng.shuffle(pos); rng.shuffle(neg)
    out=pos[:n]+neg[:n]; rng.shuffle(out); return out

train_rows=balance(train_rows, MAX_TRAIN_VIDEOS)
val_rows=balance(val_rows, MAX_VAL_VIDEOS)
print("selected train:",len(train_rows),"val:",len(val_rows))

def download_rows(rows):
    """Download selected videos from the current UCF-Crime ZIP layout.

    NOTE: HF no longer exposes individual MP4 files at /resolve/main/<video-path>.
    The repository now contains ZIP archives, so hf_hub_download must target the
    archive and we extract only the requested members.
    """
    import zipfile
    archive_paths={}
    # The action-recognition test split is in the anomaly archives; normal videos
    # come from the normal training/testing archives. We try archives lazily.
    for key,filename in UCF_ARCHIVES.items():
        needed=[rel for rel,_ in rows if (rel.startswith("Shoplifting/") or rel.startswith("Stealing/")) and key.startswith("anomaly")]
        needed += [rel for rel,_ in rows if "Normal_Videos" in rel and key.startswith("normal")]
        if not needed: continue
        try:
            archive_paths[key]=hf_hub_download(repo_id=UCF_REPO,filename=filename,repo_type="dataset",local_dir=str(ROOT/"archives"))
        except Exception as e:
            print("archive unavailable",filename,"->",type(e).__name__,str(e)[:120])
    manifest=[]
    for rel,label in tqdm(rows,desc="Extracting selected videos"):
        if label==1:
            candidates=[k for k in archive_paths if k.startswith("anomaly")]
        else:
            candidates=[k for k in archive_paths if k.startswith("normal")]
        found=False
        for key in candidates:
            zpath=archive_paths[key]
            try:
                with zipfile.ZipFile(zpath) as z:
                    names=z.namelist()
                    target=rel if rel in names else next((n for n in names if n.endswith("/"+rel) or n.endswith(rel)),None)
                    if target is None: continue
                    out=VIDEO_DIR/Path(rel).name
                    if not out.exists():
                        with z.open(target) as src, open(out,"wb") as dst:
                            import shutil; shutil.copyfileobj(src,dst)
                    vr=VideoReader(str(out),ctx=cpu(0))
                    if len(vr)>=2:
                        manifest.append({"path":str(out),"label":label,"rel":rel,"frames":len(vr)})
                        found=True; break
            except Exception as e:
                print("extract failed",rel,"from",key,"->",type(e).__name__,str(e)[:100])
        if not found:
            print("MISSING:",rel)
    return manifest


manifest = download_rows(train_rows) + download_rows(val_rows)
manifest_df=pd.DataFrame(manifest)
manifest_df["split"]="train"
val_set={r[0] for r in val_rows}
manifest_df.loc[manifest_df.rel.isin(val_set),"split"]="val"
manifest_df.to_csv(ROOT/"manifest.csv",index=False)
print(manifest_df.groupby(["split","label"]).size())


## 2. Shared temporal sampling + separate teacher/student preprocessing

Both networks see the same sampled raw RGB frames. The teacher is resized to 224×224 and normalized for VideoMAE; the student is resized to 172×172 and kept in [0,1] for MoViNet. This avoids accidentally feeding student-normalized tensors into the teacher.

In [ ]:
def sample_indices(n, k, start=None):
    if n < 2: raise ValueError("video has fewer than 2 frames")
    if n >= k:
        if start is None: start=random.randint(0,n-k)
        return np.arange(start,start+k)
    return np.linspace(0,n-1,k).round().astype(int)

def read_clip(path, indices):
    vr=VideoReader(path,ctx=cpu(0))
    arr=vr.get_batch(indices).asnumpy()  # T,H,W,C uint8
    return torch.from_numpy(arr).permute(0,3,1,2).float()/255.0

class DistillDataset(Dataset):
    def __init__(self, df, train): self.df=df.reset_index(drop=True); self.train=train
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        r=self.df.iloc[i]; vr=VideoReader(r.path,ctx=cpu(0)); n=len(vr)
        idx=sample_indices(n,NUM_FRAMES) if self.train else sample_indices(n,NUM_FRAMES, max(0,(n-NUM_FRAMES)//2))
        raw=read_clip(r.path,idx)
        if self.train and random.random()<0.5: raw=torch.flip(raw,dims=[3])
        return raw, int(r.label)

def student_preprocess(raw):
    x=F.interpolate(raw, size=(STUDENT_SIZE,STUDENT_SIZE), mode="bilinear", align_corners=False)
    return x.permute(0,1,2,3)  # T,C,H,W

teacher_processor=AutoImageProcessor.from_pretrained(TEACHER_ID)
print("teacher processor loaded")

def teacher_preprocess_batch(raw_batch):
    # raw_batch: B,T,C,H,W in [0,1]. Processor handles resize/normalization.
    clips=[]
    for clip in raw_batch:
        frames=[(f.permute(1,2,0).clamp(0,1).numpy()*255).astype(np.uint8) for f in clip]
        out=teacher_processor(frames, return_tensors="pt")
        clips.append(out["pixel_values"].squeeze(0))
    return torch.stack(clips).to(DEVICE)

def collate(batch):
    raws=torch.stack([x[0] for x in batch]); y=torch.tensor([x[1] for x in batch],dtype=torch.long)
    return raws,y

train_df=manifest_df[manifest_df.split=="train"].reset_index(drop=True)
val_df=manifest_df[manifest_df.split=="val"].reset_index(drop=True)
train_loader=DataLoader(DistillDataset(train_df,True),batch_size=TRAIN_BATCH,shuffle=True,num_workers=NUM_WORKERS,pin_memory=True,collate_fn=collate)
val_loader=DataLoader(DistillDataset(val_df,False),batch_size=TRAIN_BATCH,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,collate_fn=collate)


## 3. Load and fine-tune the VideoMAE teacher

In [ ]:
teacher=AutoModelForVideoClassification.from_pretrained(
    TEACHER_ID, num_labels=2, ignore_mismatched_sizes=True,
    label2id=LABEL2ID, id2label=ID2LABEL
).to(DEVICE)
print("teacher parameters:",sum(p.numel() for p in teacher.parameters())/1e6,"M")

def run_teacher_epoch(model, loader, optimizer=None):
    training=optimizer is not None; model.train(training)
    losses=[]; ys=[]; ps=[]
    scaler=torch.amp.GradScaler("cuda",enabled=True)
    for raw,y in tqdm(loader,leave=False,desc="teacher"):
        y=y.to(DEVICE)
        x=teacher_preprocess_batch(raw)
        with torch.amp.autocast("cuda",dtype=torch.float16):
            logits=model(pixel_values=x).logits
            loss=F.cross_entropy(logits,y)
        if training:
            optimizer.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(optimizer); scaler.update()
        losses.append(loss.item()); ys.extend(y.detach().cpu().numpy()); ps.extend(logits.argmax(1).detach().cpu().numpy())
    return float(np.mean(losses)), accuracy_score(ys,ps), f1_score(ys,ps,zero_division=0)

teacher_opt=torch.optim.AdamW(teacher.parameters(),lr=TEACHER_LR,weight_decay=WEIGHT_DECAY)
teacher_history=[]
for epoch in range(1,TEACHER_EPOCHS+1):
    tr=run_teacher_epoch(teacher,train_loader,teacher_opt)
    va=run_teacher_epoch(teacher,val_loader,None)
    teacher_history.append([epoch,*tr,*va])
    print(f"teacher epoch {epoch}: train loss={tr[0]:.4f} acc={tr[1]:.3f} f1={tr[2]:.3f} | val loss={va[0]:.4f} acc={va[1]:.3f} f1={va[2]:.3f}")
torch.save({"state_dict":teacher.state_dict(),"model_id":TEACHER_ID,"labels":ID2LABEL},CKPT_DIR/"videomae_teacher.pt")


## 4. Freeze teacher and distill into MoViNet-A0

In [ ]:
student=AutoModelForVideoClassification.from_pretrained(
    STUDENT_ID, trust_remote_code=True, num_labels=2, ignore_mismatched_sizes=True,
    label2id=LABEL2ID, id2label=ID2LABEL
).to(DEVICE)
for p in teacher.parameters(): p.requires_grad=False
teacher.eval()
print("student parameters:",sum(p.numel() for p in student.parameters())/1e6,"M")

student_opt=torch.optim.AdamW(student.parameters(),lr=STUDENT_LR,weight_decay=WEIGHT_DECAY)
scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(student_opt,T_max=DISTILL_EPOCHS)
scaler=torch.amp.GradScaler("cuda",enabled=True)
history=[]

def kd_loss(student_logits, teacher_logits, y):
    hard=F.cross_entropy(student_logits,y)
    T=KD_TEMPERATURE
    soft=F.kl_div(F.log_softmax(student_logits/T,dim=-1),F.softmax(teacher_logits/T,dim=-1),reduction="batchmean")*(T*T)
    return KD_ALPHA*hard+KD_BETA*soft, hard.detach(), soft.detach()

def eval_student(model, loader):
    model.eval(); ys=[]; probs=[]; losses=[]
    with torch.inference_mode():
        for raw,y in tqdm(loader,leave=False,desc="student eval"):
            x=student_preprocess(raw).to(DEVICE,non_blocking=True); y=y.to(DEVICE)
            with torch.amp.autocast("cuda",dtype=torch.float16): logits=model(pixel_values=x).logits
            losses.append(F.cross_entropy(logits,y).item()); probs.extend(logits.softmax(-1)[:,1].float().cpu().numpy()); ys.extend(y.cpu().numpy())
    ys=np.asarray(ys); probs=np.asarray(probs); pred=(probs>=0.5).astype(int)
    return {"loss":float(np.mean(losses)),"accuracy":accuracy_score(ys,pred),"balanced_accuracy":balanced_accuracy_score(ys,pred),"precision":precision_score(ys,pred,zero_division=0),"recall":recall_score(ys,pred,zero_division=0),"f1":f1_score(ys,pred,zero_division=0),"roc_auc":roc_auc_score(ys,probs) if len(np.unique(ys))>1 else float("nan"),"average_precision":average_precision_score(ys,probs) if len(np.unique(ys))>1 else float("nan"),"y":ys,"p":probs}

best_f1=-1
for epoch in range(1,DISTILL_EPOCHS+1):
    student.train(); total=hard_total=soft_total=0.0; steps=0
    for raw,y in tqdm(train_loader,desc=f"distill {epoch}"):
        y=y.to(DEVICE); sx=student_preprocess(raw).to(DEVICE,non_blocking=True); tx=teacher_preprocess_batch(raw)
        with torch.no_grad(), torch.amp.autocast("cuda",dtype=torch.float16): teacher_logits=teacher(pixel_values=tx).logits
        with torch.amp.autocast("cuda",dtype=torch.float16):
            student_logits=student(pixel_values=sx).logits
            loss,hard,soft=kd_loss(student_logits,teacher_logits,y)
        student_opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(student_opt)
        torch.nn.utils.clip_grad_norm_(student.parameters(),1.0); scaler.step(student_opt); scaler.update()
        total+=loss.item(); hard_total+=hard.item(); soft_total+=soft.item(); steps+=1
    scheduler.step(); m=eval_student(student,val_loader)
    history.append([epoch,total/max(steps,1),hard_total/max(steps,1),soft_total/max(steps,1),m["loss"],m["f1"],m["accuracy"],m["roc_auc"]])
    print(f"epoch {epoch}: kd={total/max(steps,1):.4f} hard={hard_total/max(steps,1):.4f} soft={soft_total/max(steps,1):.4f} | val f1={m['f1']:.3f} acc={m['accuracy']:.3f}")
    if m["f1"]>best_f1:
        best_f1=m["f1"]; torch.save({"state_dict":student.state_dict(),"model_id":STUDENT_ID,"temperature":KD_TEMPERATURE,"alpha":KD_ALPHA,"beta":KD_BETA},CKPT_DIR/"movinet_distilled_best.pt")


## 5. Threshold selection, metrics, and visual diagnostics

In [ ]:
student.load_state_dict(torch.load(CKPT_DIR/"movinet_distilled_best.pt",map_location=DEVICE)["state_dict"])
m=eval_student(student,val_loader); y=m.pop("y"); p=m.pop("p")
thresholds=np.linspace(0.05,0.95,181); f1s=[f1_score(y,p>=t,zero_division=0) for t in thresholds]; threshold=float(thresholds[int(np.argmax(f1s))])
pred=(p>=threshold).astype(int)
final={"threshold":threshold,"accuracy":accuracy_score(y,pred),"balanced_accuracy":balanced_accuracy_score(y,pred),"precision":precision_score(y,pred,zero_division=0),"recall":recall_score(y,pred,zero_division=0),"f1":f1_score(y,pred,zero_division=0),"roc_auc":roc_auc_score(y,p),"average_precision":average_precision_score(y,p)}
print(json.dumps(final,indent=2)); print(classification_report(y,pred,target_names=["normal","shoplifting"],digits=4))

cm=confusion_matrix(y,pred)
plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt="d",xticklabels=["normal","shoplifting"],yticklabels=["normal","shoplifting"]); plt.xlabel("Predicted"); plt.ylabel("True"); plt.title("Distilled student confusion matrix"); plt.tight_layout(); plt.savefig(FIG_DIR/"confusion_matrix.png",dpi=160); plt.show()
fpr,tpr,_=roc_curve(y,p); plt.figure(figsize=(5,4)); plt.plot(fpr,tpr); plt.plot([0,1],[0,1],"--"); plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.title(f"ROC AUC={final['roc_auc']:.3f}"); plt.tight_layout(); plt.savefig(FIG_DIR/"roc.png",dpi=160); plt.show()
prec,rec,_=precision_recall_curve(y,p); plt.figure(figsize=(5,4)); plt.plot(rec,prec); plt.xlabel("Recall"); plt.ylabel("Precision"); plt.title(f"PR AP={final['average_precision']:.3f}"); plt.tight_layout(); plt.savefig(FIG_DIR/"pr.png",dpi=160); plt.show()

hist=pd.DataFrame(history,columns=["epoch","kd_loss","hard_ce","soft_kl","val_loss","val_f1","val_accuracy","val_roc_auc"]); hist.to_csv(ROOT/"distillation_history.csv",index=False)
plt.figure(figsize=(6,4)); plt.plot(hist.epoch,hist.kd_loss,marker="o",label="KD loss"); plt.plot(hist.epoch,hist.val_loss,marker="o",label="Val CE"); plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.tight_layout(); plt.savefig(FIG_DIR/"loss.png",dpi=160); plt.show()


## 6. Save the deployment artifact and benchmark the lightweight student

In [ ]:
deployment={"model_id":STUDENT_ID,"state_dict":student.state_dict(),"labels":ID2LABEL,"num_frames":NUM_FRAMES,"image_size":STUDENT_SIZE,"threshold":threshold,"metrics":final,"distillation":{"teacher":TEACHER_ID,"temperature":KD_TEMPERATURE,"alpha":KD_ALPHA,"beta":KD_BETA}}
torch.save(deployment,CKPT_DIR/"movinet_distilled_deployment.pt")
with open(ROOT/"distilled_metrics.json","w") as f: json.dump(final,f,indent=2)

def predict_video(path):
    vr=VideoReader(path,ctx=cpu(0)); n=len(vr); clips=[]
    starts=np.linspace(0,max(0,n-NUM_FRAMES),VAL_CLIPS).round().astype(int)
    for s in starts:
        raw=read_clip(path,sample_indices(n,NUM_FRAMES,int(s))); clips.append(student_preprocess(raw))
    x=torch.stack(clips).to(DEVICE)
    with torch.inference_mode(),torch.amp.autocast("cuda",dtype=torch.float16): probs=student(pixel_values=x).logits.softmax(-1)[:,1]
    score=float(probs.mean().item()); return {"score":score,"prediction":"shoplifting" if score>=threshold else "normal","threshold":threshold}

sample_path=val_df.iloc[0].path if len(val_df) else train_df.iloc[0].path
print("sample prediction:",predict_video(sample_path))

dummy=torch.rand(TRAIN_BATCH,NUM_FRAMES,3,STUDENT_SIZE,STUDENT_SIZE,device=DEVICE)
for _ in range(5):
    with torch.inference_mode(),torch.amp.autocast("cuda",dtype=torch.float16): _=student(pixel_values=dummy).logits
torch.cuda.synchronize(); t0=time.perf_counter()
iters=30
for _ in range(iters):
    with torch.inference_mode(),torch.amp.autocast("cuda",dtype=torch.float16): _=student(pixel_values=dummy).logits
torch.cuda.synchronize(); elapsed=time.perf_counter()-t0
print({"batch_size":TRAIN_BATCH,"batch_latency_ms":elapsed/iters*1000,"clip_latency_ms":elapsed/iters/TRAIN_BATCH*1000,"clips_per_sec":iters*TRAIN_BATCH/elapsed})


## Notes for the experiment
- The teacher is used only during training; deployment uses the smaller MoViNet student.
- `KD_ALPHA` controls hard-label CE and `KD_BETA` controls teacher-student KL divergence. Keep them summing to 1 for the default interpretation.
- For a fast smoke test, set `MAX_TRAIN_VIDEOS=40`, `MAX_VAL_VIDEOS=20`, `TEACHER_EPOCHS=1`, and `DISTILL_EPOCHS=1` before running all.
- For a serious experiment, keep the identity-disjoint source split and evaluate on a held-out retail-specific dataset before deployment. UCF-Crime is useful for the binary benchmark but is not a substitute for a retail validation set.
